MongoDB Chat Agent with MCP
---------------------------

**Goal:** build AI application that bridges the gap between natural language and structured data using industry-standard protocols.

**Purpose:** an AI agent that lets you chat with your data using plain English, powered by LangGraph, the Model Context Protocol (MCP), and Gradio.

## Requirements

- Connect to MongoDB securely using the Model Context Protocol (MCP), an open standard for AI.
- Build a stateful agent with LangGraph that can reason and act.
- Leverage a pre-built ReAct agent to think, execute tools, and find answers.
- Wrap everything in a user-friendly UI with Gradio.

## Tech stack

- **LangGraph:** An extension of LangChain for building complex, stateful AI applications. It uses graphs to create cycles, which are essential for agents that need to think, act, and re-evaluate, just like a human would.
- **ReAct Agent:** A specific agent design that stands for “Reason and Act.” It works by generating a thought process, choosing a tool (like a database query), executing it, and observing the result to decide its next step.
- **Model Context Protocol (MCP):** An open-source standard for connecting large language models to external tools and data sources. It acts as a universal language, allowing any compliant AI to communicate with any compliant tool.
- **Gradio:** A Python library that makes it incredibly simple to create and share interactive web UIs for machine learning models and AI agents.

# Dependencies

Install necessary Python libraries.

In [ ]:
!pip install langchain-community langgraph langchain_openai langchain_mcp_adapters gradio "npx" --user
!pip install --upgrade langchain

In [ ]:
# imports

import gradio as gr
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
import os

# Setup

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# Functions

## Step 1: Connecting to MongoDB via an MCP Client

First step is to set up the `MultiServerMCPClient`. This client `langchain_mcp_adapters` is responsible for starting and managing the `mongodb-mcp-server` process, which speaks the Model Context Protocol.

In [ ]:
# This client will manage the connection to the MongoDB MCP server.
client = MultiServerMCPClient(
    {
        "MongoDB": {
            # "stdio" means the agent communicates with the server process
            # via standard input/output.
            "transport": "stdio",
            # "npx" is used to run the server package without a global installation.
            "command": "npx",
            "args": [
                "-y",
                "mongodb-mcp-server",
                "--connectionString",
                # IMPORTANT: Replace this with your actual MongoDB connection string.
                "<mongo db connection string>"
            ]
        }
    }
)

## Step 2: Fetch Tools from MCP Server

In [ ]:
tools = await client.get_tools()

## Step 3: Defining the ReAct Agent in LangGraph

Define the “brain” of our application — the ReAct agent. We initialize our chosen language model (`gpt-4o-mini`) and then create the agent within our main chat function. This is because the tools the agent can use are fetched asynchronously from the MCP client.

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")
agent = create_react_agent(model=model, tools=tools)

## Step 4: Building a Web Interface with Gradio

To make our agent interactive, we’ll use Gradio’s `ChatInterface`. We will define a single async function, `chat_function`, that Gradio will call every time a user sends a message.

In [ ]:
iface = gr.ChatInterface(
    fn=chat_function,
    title="LangGraph MongoDB Agent",
    description="Ask your questions about the MongoDB collections. For example: 'Tell me all mongo collections'",
    examples=[["Tell me all mongo collections"], ["List all databases"]],
    cache_examples=False,

# The Complete Code

Full, commented script in `mongodb-chat.py`.

To start your agent, run the script from your terminal:

`python app.py`

Gradio will provide a local URL (e.g., http://127.0.0.1:7860). Open it in your browser to start chatting with your database.